# Kalman AI — the first AI-controlled application

This is the first application built on the **control contract** introduced in §8 of
`00_Kalman101.ipynb`: a controller is just a callable `(x̂, P, t, ν) → u`, and the `KalmanFilter`
never knows what's behind it — a fixed gain, a PID, an optimal LQR solve, or a trained neural network.

A Kalman filter **estimates** the hidden state of a system from noisy measurements.  But estimation alone
does not *change* the system — it only watches.  **Control** closes the loop: a controller reads the
filter's state estimate $\hat{x}$ and computes a control input $u$ that steers the system toward a
reference $x_{\mathrm{ref}}$.

This notebook demonstrates the framework's **pluggable control layer** — from a one-line proportional
gain to a neural policy trained offline via MPPI on the model's own physics.

### Architecture

```
filter.forward(zk, uk=controller)

  each step k:
  ┌─────────────────────────────────────────────────────────┐
  │ 1. u_k = controller(x̂_k, P_k, t_k, nu=ν_{k-1})       │  ← callable
  │ 2. engine._predict(model, x, P, u=u_k, t=t_k)          │  ← x⁻ = f(x) + B·u
  │ 3. engine._update(model, x, P, z_k, t=t_k)             │  ← classical Kalman gain
  └─────────────────────────────────────────────────────────┘
```

**Key design principle:** the `KalmanFilter` has zero knowledge of `KalmanController`.  The filter
only sees `uk` as a callable `(x̂, P, t, ν) → u`.  All estimation machinery (F, H, K, gain
computation) stays classical and untouched.

**Genuine closed loop:** every controller demo below drives a *true* simulated trajectory with the
*exact* `u` the controller computed from the filter's estimate — not an open-loop placeholder. See
`closed_loop_run()` after §1.

### Two kinds of Q and R

| Symbol | Meaning | Owned by |
|--------|---------|----------|
| Qnoise, Rnoise | Kalman filter noise covariances | model |
| Qcost, Rcost | Controller cost weights (penalise error vs effort) | controller |

In [ ]:
%pip install numpy matplotlib scipy > /dev/null
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

import kalman; kalman.set_debug(0)
from kalman.filters import ExtendedKalmanFilter
from kalman.apps.vehicles import CarLaneModel
from kalman.framework import (
    ProportionalKalmanController,
    PIDKalmanController,
    LQRKalmanController,
    NeuralKalmanController,
    BoostKalmanController,
    AdaptiveQRKalmanController,
)

## 1 — The test system: lateral lane-keeping

A car drives at constant speed $v$ along a lane.  Its state is:

$$\mathbf{x} = \begin{bmatrix} y \\ \psi \\ v \end{bmatrix}$$

where $y$ is the lateral offset from lane centre [m], $\psi$ is the heading angle [rad], and $v$ is
the longitudinal speed [m/s].  The single control input $u = [\delta]$ is the **steering angle** [rad].

Bicycle kinematics (Euler-discretised):

$$y_{k+1} = y_k + v_k \sin\psi_k \, \Delta t, \qquad
  \psi_{k+1} = \psi_k + \frac{v_k \tan\delta_k}{L} \Delta t, \qquad
  v_{k+1} = v_k$$

A noisy GPS measures lateral position only: $z = y + \text{noise}$.

The goal: steer $y \to 0$ (lane centre) from an initial offset.

In [ ]:
np.random.seed(42)

model = CarLaneModel("GPS", fs=50, x0=np.array([[1.5], [0.1], [15.0]]))
print(f"State dim n={model.n}, Measurement dim m={model.m}, Control dim l={model.l}")
print(f"Initial state: y={model.x0[0,0]:.1f} m, ψ={model.x0[1,0]:.2f} rad, v={model.x0[2,0]:.0f} m/s")

N = 250   # 5 seconds at 50 Hz
dt = 1.0 / model.fs

# Simulate noisy GPS measurements of a car drifting without control
x_true = model.x0.copy()
zk_open = np.zeros((N, 1))
xk_open = np.zeros((N, 3))
for k in range(N):
    xk_open[k] = x_true.ravel()
    zk_open[k] = x_true[0, 0] + np.random.normal(0, 0.3)
    x_true = model.f(x_true, u=np.array([[0.0]]), t=k * dt)

fig, ax = plt.subplots(figsize=(11, 3.5))
t = np.arange(N) * dt
ax.scatter(t, zk_open, s=8, alpha=0.3, color="tab:orange", label="GPS (noisy)")
ax.plot(t, xk_open[:, 0], color="gray", lw=1.5, label="True lateral offset")
ax.axhline(0, color="black", ls="--", lw=0.8, label="Lane centre")
ax.set_xlabel("time [s]"); ax.set_ylabel("lateral offset [m]")
ax.set_title("Open-loop: the car drifts away from the lane")
ax.legend(); ax.grid(True, ls=":", alpha=0.6); plt.show()

### Closing the loop, honestly

A controller computes $u$ from the filter's **estimate** $\hat{x}$, not from the truth — the
true system only ever sees the $u$ the controller actually chose. To test a controller fairly we must
simulate the true trajectory **in lockstep** with the filter: at each step, ask the controller for
$u$ given the filter's current $(\hat{x}, P)$, advance the *true* state with `model.f(x_true, u=u)`
(plus process noise), measure it, and feed that measurement back into the same filter's predict/update.

`closed_loop_run()` below does exactly this — every controller comparison in this notebook uses it.

In [ ]:
def closed_loop_run(controller, N, seed=0):
    """Simulate the true system in genuine closed loop with `controller`.

    At each step: u = controller(filter's current x̂, P) -> the TRUE state is advanced
    with that same u via model.f() (+ process noise) -> the resulting noisy measurement
    is fed back into the same filter's predict/update. Returns (record, xk_true, zk_hist).
    """
    rng = np.random.default_rng(seed)
    model = controller.model
    ekf = ExtendedKalmanFilter(model)
    ekf.reset()

    x_true = model.x0.copy()
    dt_ = 1.0 / model.fs
    nu = np.zeros((model.m, 1))
    t_ = 0.0
    q_std = np.sqrt(np.diag(model.Q()))
    r_std = np.sqrt(np.diag(model.R()))

    xk_true = np.zeros((N, model.n))
    zk_hist = np.zeros((N, model.m))

    for k in range(N):
        u = controller(ekf.x, ekf.P, t_, nu=nu)

        x_true = np.asarray(model.f(x_true, u=u, t=t_), dtype=float).reshape(-1, 1)
        x_true = x_true + rng.normal(0, q_std).reshape(-1, 1)
        z = np.asarray(model.h(x_true, t=t_), dtype=float).reshape(-1, 1)
        z = z + rng.normal(0, r_std).reshape(-1, 1)

        ekf.measurement.append(z)
        ekf.control.append(u.ravel())
        ekf.x, ekf.P, F = ekf.engine._predict(model, ekf.x, ekf.P, u=u, t=t_)
        ekf.prior.append((ekf.x, ekf.P))
        ekf.transition.append(F)

        h_pred = model.h(ekf.x, t=t_)
        nu = z - np.asarray(h_pred, dtype=float).reshape(-1, 1)
        ekf.innovation.append(nu)

        ekf.x, ekf.P = ekf.engine._update(model, ekf.x, ekf.P, z, t=t_)
        ekf.posterior.append((ekf.x, ekf.P))

        xk_true[k] = x_true.ravel()
        zk_hist[k] = z.ravel()
        t_ += dt_

    return ekf.get_record(), xk_true, zk_hist

## 2 — Proportional controller (baseline)

The simplest controller: $u = -K_p \cdot (\hat{x} - x_{\mathrm{ref}})$.  One gain matrix, no training,
no scipy.  Works for any model with $l > 0$.

We pass the controller as a **callable** to `filter.forward(zk, uk=controller)` — the filter calls it
at each step with the current state estimate, covariance, time, and the previous innovation.

In [ ]:
model_p = CarLaneModel("GPS", fs=50, x0=np.array([[1.5], [0.1], [15.0]]))
Kp = np.array([[0.8, 0.5, 0.0]])   # steer based on lateral offset and heading; ignore speed
ctrl_p = ProportionalKalmanController(model_p, Kp=Kp)

rec_p, xk_true_p, zk_ctrl = closed_loop_run(ctrl_p, N, seed=1)
est_p = np.asarray(rec_p.xk)
t = np.arange(N) * dt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(t, zk_ctrl, s=6, alpha=0.2, color="tab:orange", label="GPS")
axes[0].plot(t, xk_true_p[:, 0], color="gray", lw=1, ls="--", label="True (closed loop)")
axes[0].plot(t, est_p[:, 0], color="steelblue", lw=1.5, label="EKF estimate")
axes[0].axhline(0, color="black", ls="--", lw=0.8)
axes[0].set_title("Lateral offset"); axes[0].set_ylabel("y [m]")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.6)

axes[1].plot(t, np.degrees(est_p[:, 1]), color="seagreen", lw=1.2)
axes[1].set_title("Heading"); axes[1].set_ylabel("ψ [deg]")
axes[1].grid(True, ls=":", alpha=0.6)

costs = [c for _, c in ctrl_p.cost_log]
axes[2].plot(t, costs, color="crimson", lw=1.2)
axes[2].set_title("Controller cost"); axes[2].set_ylabel("J")
axes[2].grid(True, ls=":", alpha=0.6)

for ax in axes: ax.set_xlabel("time [s]")
fig.suptitle("Proportional controller — genuine closed loop (truth responds to u)", y=1.02)
plt.tight_layout(); plt.show()
print(f"Final true lateral offset: {xk_true_p[-1, 0]:.4f} m")
print(f"Total cost: {sum(costs):.2f}")

## 3 — PID controller

Adds integral and derivative terms: $u = -K_p \, e - K_i \int e \, dt - K_d \, \dot{e}$.

The integral term eliminates steady-state error; the derivative damps overshoot.  Set any gain to 0
for P-only, PD, PI, etc.

In [ ]:
model_pid = CarLaneModel("GPS", fs=50, x0=np.array([[1.5], [0.1], [15.0]]))
ctrl_pid = PIDKalmanController(
    model_pid,
    Kp=np.array([[0.8, 0.5, 0.0]]),
    Ki=np.array([[0.05, 0.0, 0.0]]),   # integral on lateral offset only
    Kd=np.array([[0.2, 0.1, 0.0]]),    # derivative damping
)

rec_pid, xk_true_pid, _ = closed_loop_run(ctrl_pid, N, seed=1)
est_pid = np.asarray(rec_pid.xk)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, est_p[:, 0], color="steelblue", lw=1.2, label="Proportional")
ax.plot(t, est_pid[:, 0], color="crimson", lw=1.5, label="PID")
ax.axhline(0, color="black", ls="--", lw=0.8, label="Lane centre")
ax.set_xlabel("time [s]"); ax.set_ylabel("lateral offset [m]")
ax.set_title("P vs PID — lateral lane-keeping (closed loop, same noise seed)")
ax.legend(); ax.grid(True, ls=":", alpha=0.6); plt.show()

cost_p = sum(c for _, c in ctrl_p.cost_log)
cost_pid = sum(c for _, c in ctrl_pid.cost_log)
print(f"Total cost — Proportional: {cost_p:.2f},  PID: {cost_pid:.2f}")

## 4 — LQR controller (optimal for linear systems)

The **Linear Quadratic Regulator** computes the optimal gain $K$ from the Discrete Algebraic Riccati
Equation (DARE).  For a linear system, this is the *best possible* constant-gain controller — it
minimises $J = \sum e^\top Q_{\mathrm{cost}} \, e + u^\top R_{\mathrm{cost}} \, u$ exactly.

It requires an explicit linear model ($F$ and $B$ matrices), so it works as a theoretical
baseline — the neural controller should try to match or approach it.

In [ ]:
model_lqr = CarLaneModel("GPS", fs=50, x0=np.array([[1.5], [0.1], [15.0]]))
Qcost, Rcost, x_ref = model_lqr.control_cost()
ctrl_lqr = LQRKalmanController(model_lqr, Qcost=Qcost, Rcost=Rcost, x_ref=x_ref)
ctrl_lqr.fit()
print(f"LQR gain K = {ctrl_lqr.K.ravel()}")

rec_lqr, xk_true_lqr, _ = closed_loop_run(ctrl_lqr, N, seed=1)
est_lqr = np.asarray(rec_lqr.xk)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, est_p[:, 0], color="steelblue", lw=1, alpha=0.7, label="Proportional")
ax.plot(t, est_pid[:, 0], color="crimson", lw=1, alpha=0.7, label="PID")
ax.plot(t, est_lqr[:, 0], color="darkgreen", lw=2, label="LQR (optimal)")
ax.axhline(0, color="black", ls="--", lw=0.8)
ax.set_xlabel("time [s]"); ax.set_ylabel("lateral offset [m]")
ax.set_title("P vs PID vs LQR — lateral lane-keeping (closed loop, same noise seed)")
ax.legend(); ax.grid(True, ls=":", alpha=0.6); plt.show()

cost_lqr = sum(c for _, c in ctrl_lqr.cost_log)
print(f"Total cost — P: {cost_p:.2f},  PID: {cost_pid:.2f},  LQR: {cost_lqr:.2f}")

## 5 — Neural controller (MPPI-trained)

The `NeuralKalmanController` trains a 2-layer MLP offline via **Model Predictive Path Integral (MPPI)**
rollouts of `model.f()` — the physics simulation already in the framework doubles as the training
environment.  No external simulator needed.

The MLP sees $[\hat{x} - x_{\mathrm{ref}},\; \sqrt{\mathrm{diag}(P)},\; \nu]$ — never $F$, $H$, or raw $z$.

### MPPI algorithm

1. Sample $N$ random control perturbation sequences $\delta u$
2. Roll each through `model.f(x, u+δu, t)` (full nonlinear physics)
3. Compute cost $J = \sum e^\top Q_{\mathrm{cost}} \, e + u^\top R_{\mathrm{cost}} \, u$ per trajectory
4. Weight: $w_i = \exp(-J_i / \lambda)$, normalised
5. Optimal: $u^*_k = \sum w_i \cdot (u_k + \delta u_k^i)$
6. Collect $(\text{state}, u^*)$ dataset → train MLP

In [ ]:
model_nn = CarLaneModel("GPS", fs=50, x0=np.array([[1.5], [0.1], [15.0]]))
Qcost, Rcost, x_ref = model_nn.control_cost()
ctrl_nn = NeuralKalmanController(
    model_nn, hidden=32,
    Qcost=Qcost, Rcost=Rcost, x_ref=x_ref,
)

print("Training neural controller via MPPI...")
ctrl_nn.fit_mppi(n_rollouts=300, horizon=50, noise_sigma=0.3, lam=1.0,
                 epochs=200, lr=5e-4, seed=42)
print("Done.")

rec_nn, xk_true_nn, _ = closed_loop_run(ctrl_nn, N, seed=1)
est_nn = np.asarray(rec_nn.xk)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(t, est_p[:, 0], color="steelblue", lw=1, alpha=0.5, label="Proportional")
axes[0].plot(t, est_lqr[:, 0], color="darkgreen", lw=1.2, alpha=0.7, label="LQR")
axes[0].plot(t, est_nn[:, 0], color="darkorange", lw=2, label="Neural (MPPI)")
axes[0].axhline(0, color="black", ls="--", lw=0.8)
axes[0].set_xlabel("time [s]"); axes[0].set_ylabel("y [m]")
axes[0].set_title("Lateral offset comparison (closed loop, same noise seed)")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.6)

axes[1].plot(t, [c for _, c in ctrl_p.cost_log], color="steelblue", lw=1, alpha=0.5, label="P")
axes[1].plot(t, [c for _, c in ctrl_lqr.cost_log], color="darkgreen", lw=1.2, alpha=0.7, label="LQR")
axes[1].plot(t, [c for _, c in ctrl_nn.cost_log], color="darkorange", lw=2, label="Neural")
axes[1].set_xlabel("time [s]"); axes[1].set_ylabel("cost J")
axes[1].set_title("Per-step cost comparison")
axes[1].legend(fontsize=8); axes[1].grid(True, ls=":", alpha=0.6)

plt.tight_layout(); plt.show()

cost_nn = sum(c for _, c in ctrl_nn.cost_log)
print(f"Total cost — P: {cost_p:.2f},  PID: {cost_pid:.2f},  LQR: {cost_lqr:.2f},  Neural: {cost_nn:.2f}")

## 5b — Performance comparison: P → PID → LQR → Neural

All four controllers ran on the same car model, same initial condition, same noise seed.
This cell collects the results and compares them head-to-head.

In [ ]:
cost_p   = sum(c for _, c in ctrl_p.cost_log)
cost_pid = sum(c for _, c in ctrl_pid.cost_log)
cost_lqr = sum(c for _, c in ctrl_lqr.cost_log)
cost_nn  = sum(c for _, c in ctrl_nn.cost_log)

labels   = ["Proportional", "PID", "LQR", "Neural (MPPI)"]
costs    = [cost_p, cost_pid, cost_lqr, cost_nn]
colors   = ["steelblue", "crimson", "darkgreen", "darkorange"]
traces   = [est_p[:, 0], est_pid[:, 0], est_lqr[:, 0], est_nn[:, 0]]

# Per-step RMS lateral error for each controller
rms  = [float(np.sqrt(np.mean(tr**2)))            for tr in traces]
mean_abs = [float(np.mean(np.abs(tr)))             for tr in traces]
max_abs  = [float(np.max(np.abs(tr)))              for tr in traces]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# --- time-series overlay ---
for tr, lbl, col in zip(traces, labels, colors):
    axes[0].plot(t, tr, color=col, lw=1.5, label=lbl)
axes[0].axhline(0, color="black", ls="--", lw=0.8)
axes[0].set_xlabel("time [s]"); axes[0].set_ylabel("y [m]")
axes[0].set_title("Lateral offset — all controllers")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.6)

# --- bar charts: metric trio ---
x_pos = np.arange(4)
w = 0.25
for i, (metric, label) in enumerate([(mean_abs, "mean |y|"), (max_abs, "max |y|"), (rms, "RMS")]):
    axes[1].bar(x_pos + i*w, metric, width=w, color=[c for c in colors], alpha=0.8, label=label)
axes[1].set_xticks(x_pos + w)
axes[1].set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("lateral error [m]")
axes[1].set_title("Error metrics")
legend_patches = [plt.Rectangle((0,0),1,1, color="none",  label="mean |y| / max |y| / RMS\n(left / center / right bar)")]
axes[1].legend(handles=legend_patches, fontsize=8)
axes[1].grid(True, axis="y", ls=":", alpha=0.6)

# --- total cost bar ---
bars = axes[2].bar(labels, costs, color=colors, alpha=0.85)
axes[2].set_ylabel("total cost J")
axes[2].set_title("Total cumulative cost (lower = better)")
for bar, c in zip(bars, costs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5*max(costs)/100,
                 f"{c:.0f}", ha="center", va="bottom", fontsize=9)
axes[2].set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
axes[2].grid(True, axis="y", ls=":", alpha=0.6)

fig.suptitle("Controller comparison — P → PID → LQR → Neural (same model, same noise seed)",
             fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

print(f"{'Controller':<18} {'mean |y| [m]':>14} {'max |y| [m]':>13} {'RMS [m]':>10} {'total cost':>12}")
print("-" * 70)
for lbl, ma, mx, rm, co in zip(labels, mean_abs, max_abs, rms, costs):
    print(f"{lbl:<18} {ma:>14.4f} {mx:>13.4f} {rm:>10.4f} {co:>12.1f}")

## 6 — BoostKalmanController: gain boosting via R-scaling

When the filter is "surprised" (Normalised Innovation Squared >> $m$), the measurement is far from
what the model predicted.  **BoostKalmanController** temporarily shrinks $R$ so the Kalman gain $K$ grows — the
filter trusts measurements more and re-acquires faster.

Below, we simulate a sudden lane change (a jump in the true lateral offset) and compare the filter's
response with and without BoostKalmanController.

In [ ]:
# Simulate a sudden lane change at t=2s
N_boost = 300
zk_jump = np.zeros((N_boost, 1))
for k in range(N_boost):
    y_true = 0.0 if k < 100 else 2.0   # sudden jump at step 100 (t=2s)
    zk_jump[k] = y_true + np.random.normal(0, 0.3)

# Without BoostKalmanController
model_nb = CarLaneModel("GPS", fs=50)
ekf_nb = ExtendedKalmanFilter(model_nb)
rec_nb = ekf_nb.forward(zk_jump)

# With BoostKalmanController
model_b = CarLaneModel("GPS", fs=50)
ekf_b = ExtendedKalmanFilter(model_b)
boost = BoostKalmanController(model_b, boost_factor=5.0, threshold=2.0, window=5)

# Run step-by-step to apply BoostKalmanController
ekf_b.reset()
t_b = 0.0
dt_b = 1.0 / model_b.fs
for z in zk_jump:
    ekf_b.x, ekf_b.P, F = ekf_b.engine._predict(model_b, ekf_b.x, ekf_b.P, u=np.zeros((1,1)), t=t_b)
    ekf_b.prior.append((ekf_b.x, ekf_b.P))
    ekf_b.transition.append(F)

    h_pred = model_b.h(ekf_b.x, t=t_b)
    nu = z.reshape(-1,1) - np.asarray(h_pred).reshape(-1,1)
    H = model_b.H(x=ekf_b.x, t=t_b)
    S = H @ ekf_b.P @ H.T + model_b.R(t=t_b)
    boost(nu, S, t=t_b)

    ekf_b.x, ekf_b.P = ekf_b.engine._update(model_b, ekf_b.x, ekf_b.P, z, t=t_b)
    ekf_b.posterior.append((ekf_b.x, ekf_b.P))
    ekf_b.measurement.append(z)
    ekf_b.innovation.append(nu)
    t_b += dt_b

rec_b = ekf_b.get_record()
t_plot = np.arange(N_boost) * dt_b

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(t_plot, zk_jump, s=5, alpha=0.2, color="tab:orange", label="GPS")
axes[0].plot(t_plot, np.asarray(rec_nb.xk)[:, 0], color="steelblue", lw=1.5, label="Without BoostKalmanController")
axes[0].plot(t_plot, np.asarray(rec_b.xk)[:, 0], color="crimson", lw=1.5, label="With BoostKalmanController")
axes[0].axhline(2.0, color="black", ls="--", lw=0.8)
axes[0].set_xlabel("time [s]"); axes[0].set_ylabel("y [m]")
axes[0].set_title("Lane change re-acquisition"); axes[0].legend(fontsize=8)
axes[0].grid(True, ls=":", alpha=0.6)

scales = [s for _, s in boost.scale_log]
axes[1].plot(t_plot, scales, color="purple", lw=1.2)
axes[1].set_xlabel("time [s]"); axes[1].set_ylabel("R scale factor")
axes[1].set_title("BoostKalmanController scale log (< 1 = boosting)")
axes[1].grid(True, ls=":", alpha=0.6)

plt.tight_layout(); plt.show()

## 7 — AdaptiveQRKalmanController: innovation-based Q scaling

If the model's process noise $Q$ is mistuned, the filter becomes overconfident or overcautious.
**AdaptiveQRKalmanController** compares the empirical innovation covariance $E[\nu\nu^\top]$ (sliding window) against
the model's predicted $S = H P^- H^\top + R$.  If the ratio exceeds 1, $Q$ is too small → scale it up.

In [ ]:
# Deliberately mis-tune Q (too small by 10x) and let AdaptiveQRKalmanController correct it
model_aq = CarLaneModel("GPS", fs=50, q_lat=1e-3, q_head=1e-5, q_vel=1e-4)
ekf_aq = ExtendedKalmanFilter(model_aq)
aqr = AdaptiveQRKalmanController(model_aq, window=20, q_min=0.5, q_max=10.0)

# Run step-by-step
ekf_aq.reset()
t_aq = 0.0
for z in zk_jump:
    ekf_aq.x, ekf_aq.P, F = ekf_aq.engine._predict(model_aq, ekf_aq.x, ekf_aq.P, u=np.zeros((1,1)), t=t_aq)
    ekf_aq.prior.append((ekf_aq.x, ekf_aq.P))
    ekf_aq.transition.append(F)

    h_pred = model_aq.h(ekf_aq.x, t=t_aq)
    nu = z.reshape(-1,1) - np.asarray(h_pred).reshape(-1,1)
    H = model_aq.H(x=ekf_aq.x, t=t_aq)
    S = H @ ekf_aq.P @ H.T + model_aq.R(t=t_aq)
    aqr(nu, S, t=t_aq)

    ekf_aq.x, ekf_aq.P = ekf_aq.engine._update(model_aq, ekf_aq.x, ekf_aq.P, z, t=t_aq)
    ekf_aq.posterior.append((ekf_aq.x, ekf_aq.P))
    ekf_aq.measurement.append(z)
    ekf_aq.innovation.append(nu)
    t_aq += dt_b

rec_aq = ekf_aq.get_record()

# Also run without adaptation for comparison
model_no = CarLaneModel("GPS", fs=50, q_lat=1e-3, q_head=1e-5, q_vel=1e-4)
ekf_no = ExtendedKalmanFilter(model_no)
rec_no = ekf_no.forward(zk_jump)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(t_plot, zk_jump, s=5, alpha=0.2, color="tab:orange", label="GPS")
axes[0].plot(t_plot, np.asarray(rec_no.xk)[:, 0], color="steelblue", lw=1.2, label="Mistuned Q (no adapt)")
axes[0].plot(t_plot, np.asarray(rec_aq.xk)[:, 0], color="crimson", lw=1.5, label="AdaptiveQRKalmanController")
axes[0].set_xlabel("time [s]"); axes[0].set_ylabel("y [m]")
axes[0].set_title("AdaptiveQRKalmanController — correcting mistuned Q")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.6)

scales_q = [s for _, s in aqr.scale_log]
axes[1].plot(np.arange(len(scales_q)) * dt_b, scales_q, color="seagreen", lw=1.2)
axes[1].axhline(1.0, color="black", ls="--", lw=0.8)
axes[1].set_xlabel("time [s]"); axes[1].set_ylabel("Q scale factor")
axes[1].set_title("AdaptiveQRKalmanController scale log (> 1 = Q growing)")
axes[1].grid(True, ls=":", alpha=0.6)

plt.tight_layout(); plt.show()

## 8 — InteractiveKalmanFilter (IMM)

When a system switches between distinct regimes (atmosphere/vacuum, road/ice), a single model is
inadequate.  The **Interacting Multiple Model** filter runs $M$ models in parallel, each with its
own `KalmanFilter`, and at each step:

1. **Mixes** states across models (interaction)
2. **Filters** each model's predict + update
3. **Scores** each model by innovation likelihood
4. **Updates** mode probabilities $\mu$ (Bayesian)
5. **Combines**: $\hat{x} = \sum \mu_j \hat{x}_j$

Below, a system switches between two lateral dynamics regimes (normal road vs icy road with higher
process noise and reduced steering effectiveness).

In [ ]:
from kalman.filters import InteractiveKalmanFilter

# Two regimes: normal road and icy road
model_normal = CarLaneModel("road", fs=50, q_lat=1e-2, q_head=1e-4)
model_icy = CarLaneModel("ice", fs=50, q_lat=0.5, q_head=1e-2)  # much higher process noise

# Transition matrix: sticky regimes (high diagonal)
Pi = np.array([
    [0.98, 0.02],   # normal: 98% stay, 2% switch to ice
    [0.05, 0.95],   # icy: 5% switch back, 95% stay
])

ikf = InteractiveKalmanFilter([model_normal, model_icy], transition=Pi)

# Simulate: normal for first 150 steps, icy for next 150
N_imm = 300
zk_imm = np.zeros((N_imm, 1))
regime_true = np.zeros(N_imm)
y_true_imm = 0.0
for k in range(N_imm):
    if k < 150:
        y_true_imm += np.random.normal(0, 0.01)   # normal road
        regime_true[k] = 0
    else:
        y_true_imm += np.random.normal(0, 0.1)    # icy road — much more drift
        regime_true[k] = 1
    zk_imm[k] = y_true_imm + np.random.normal(0, 0.3)

rec_imm = ikf.forward(zk_imm)
mu = ikf.mode_probabilities

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
t_imm = np.arange(N_imm) * dt

axes[0].scatter(t_imm, zk_imm, s=5, alpha=0.25, color="tab:orange", label="GPS")
axes[0].plot(t_imm, np.asarray(rec_imm.xk)[:, 0], color="steelblue", lw=1.5, label="IMM estimate")
axes[0].axvline(3.0, color="red", ls=":", lw=1, label="Regime switch")
axes[0].set_ylabel("y [m]")
axes[0].set_title("InteractiveKalmanFilter — regime switching")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.6)

axes[1].plot(t_imm, mu[:, 0], color="steelblue", lw=1.5, label="μ(normal road)")
axes[1].plot(t_imm, mu[:, 1], color="crimson", lw=1.5, label="μ(icy road)")
axes[1].fill_between(t_imm, 0, regime_true, alpha=0.1, color="red", label="True regime (ice)")
axes[1].axvline(3.0, color="red", ls=":", lw=1)
axes[1].set_xlabel("time [s]"); axes[1].set_ylabel("mode probability")
axes[1].set_title("Mode probabilities — regime detection")
axes[1].legend(fontsize=8); axes[1].grid(True, ls=":", alpha=0.6)

plt.tight_layout(); plt.show()

## 9 — IMM Router: pluggable model routing

The standard IMM uses a **BayesianRouter**: model j gets weight proportional to
its Gaussian innovation likelihood L_j = N(ν_j; 0, S_j) × the Markov transition prior c_j.
This is optimal when all models are correctly specified Gaussians.

**Any router can be swapped in** — the IMM is agnostic to how μ is computed:

```
ikf = InteractiveKalmanFilter(models, Pi, router=BayesianRouter())              # default
ikf = InteractiveKalmanFilter(models, Pi, router=NeuralRouter(model=MLP()))     # learned
ikf = InteractiveKalmanFilter(models, Pi, router=ThresholdRouter())             # threshold
ikf = InteractiveKalmanFilter(models, Pi, router=HybridRouter(r_a, r_b, alpha)) # blend
```

The router receives three inputs at every step and returns updated mode probabilities:

| Input | Shape | Meaning |
|---|---|---|
| `nis_arr` | `(k+1, M)` | NIS history — how surprised each model was at each step |
| `c` | `(M,)` | Predicted mode probs from the Markov transition Π |
| `likelihoods` | `(M,)` | Gaussian L_j = N(ν_j; 0, S_j) per model |

`NeuralRouter` owns NIS feature extraction only — the architecture is a separate, pluggable
`model` object (e.g. `MLP(hidden=16)`) following the `model(features) → probs` convention.

### NeuralRouter — why NIS and not innovations or mode probs?

| Feature | Problem |
|---|---|
| Raw innovations ν_j | Different scales across models — hard to normalise |
| Mode probs μ | **Distribution shift**: training runs Bayesian μ, inference produces neural μ → different sequences |
| **NIS_j window** | Dimensionless, χ²(m) under correct model, **independent of routing policy** |

NIS_j = ν_j^T S_j^{-1} ν_j measures "how surprised model j is by the measurement."
Under the correct model: NIS_j ≈ 1 (well-calibrated).  Under a wrong model: NIS_j ≫ 1.


In [ ]:
from kalman.filters import InteractiveKalmanFilter
from kalman.framework.routers import BayesianRouter, ThresholdRouter, NeuralRouter, HybridRouter
from kalman.framework.routers.nn import MLP
from kalman.framework.model import KalmanModel

# ── Two 1-D models: very different process noise ────────────────────────────
#   smooth: Q = 0.1²   rough: Q = 2.0²   (same R = 1.0² for both)
#   NIS under smooth regime:  NIS_smooth ≈ 1,  NIS_rough ≈ 0.25  (overconfident)
#   NIS under rough  regime:  NIS_smooth ≫ 1,  NIS_rough ≈ 1     (correct model)

def _make_models():
    m0 = KalmanModel('smooth', F=np.eye(1), H=np.eye(1),
                     Q=np.array([[0.1**2]]), R=np.array([[1.0**2]]),
                     fs=1.0, x0=np.array([[0.0]]))
    m1 = KalmanModel('rough', F=np.eye(1), H=np.eye(1),
                     Q=np.array([[2.0**2]]), R=np.array([[1.0**2]]),
                     fs=1.0, x0=np.array([[0.0]]))
    return m0, m1

Pi_r = np.array([[0.95, 0.05], [0.05, 0.95]])

def sim_switching(regime_seq, seed=None):
    rng = np.random.default_rng(seed)
    x, zk, labels = 0.0, [], []
    for r in regime_seq:
        x += rng.normal(0, 0.1 if r == 0 else 2.0)
        zk.append([x + rng.normal(0, 1.0)]); labels.append(r)
    return np.array(zk), np.array(labels)

T = 200

# ── Training scenarios (3 × 200 steps) ──────────────────────────────────────
regimes_train = [
    np.r_[np.zeros(100,int), np.ones(100,int)],
    np.r_[np.ones(50,int), np.zeros(100,int), np.ones(50,int)],
    np.r_[np.zeros(60,int), np.ones(80,int), np.zeros(60,int)],
]
train_zk, train_lbl = [], []
for rs in regimes_train:
    z, l = sim_switching(rs, seed=len(train_zk)); train_zk.append(z); train_lbl.append(l)

# ── Test: 0 → 1 → 0 ─────────────────────────────────────────────────────────
r_test = np.r_[np.zeros(70,int), np.ones(60,int), np.zeros(70,int)]
zk_test, lbl_test = sim_switching(r_test, seed=99)

# ── Run each router ──────────────────────────────────────────────────────────
neural_router = NeuralRouter(model=MLP(hidden=16), window=5)

routers = {
    "BayesianRouter":         BayesianRouter(),
    "ThresholdRouter":        ThresholdRouter(m=1, window=5, soft=True),
    "NeuralRouter":           neural_router,
    "HybridRouter (α=0.5)":   HybridRouter(BayesianRouter(), neural_router, alpha=0.5),
}

mu_results, acc_results = {}, {}
for name, router in routers.items():
    m0, m1 = _make_models()
    ikf = InteractiveKalmanFilter([m0, m1], Pi_r, router=router)
    if name in ("NeuralRouter", "HybridRouter (α=0.5)"):
        ikf.fit(train_zk, train_lbl, epochs=500, lr=2e-3, seed=7)
    ikf.forward(zk_test)
    mu_results[name] = ikf.mode_probabilities
    acc_results[name] = float((np.argmax(ikf.mode_probabilities,1) == lbl_test).mean())

for name, acc in acc_results.items():
    print(f"{name:<30} {acc*100:.1f}%")

# ── Plot comparison ───────────────────────────────────────────────────────────
t_nim = np.arange(T)
colors = {"BayesianRouter": "steelblue", "ThresholdRouter": "seagreen",
          "NeuralRouter": "darkorange", "HybridRouter (α=0.5)": "violet"}

fig, axes = plt.subplots(len(routers) + 1, 1, figsize=(13, 3*(len(routers)+1)), sharex=True)

axes[0].scatter(t_nim, zk_test, s=6, alpha=0.3, color="tab:orange", label="measurements")
axes[0].fill_between(t_nim, -4, 4, where=(lbl_test==1), alpha=0.12,
                     color="crimson", label="true rough regime")
axes[0].set_ylabel("z"); axes[0].set_title("Measurements + true regime (red = rough)")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.5)

for ax, (name, mu) in zip(axes[1:], mu_results.items()):
    ax.plot(t_nim, mu[:, 1], color=colors[name], lw=1.5,
            label=f"{name}   acc={acc_results[name]*100:.1f}%")
    ax.fill_between(t_nim, 0, lbl_test, alpha=0.1, color="crimson")
    ax.axhline(0.5, color="black", ls="--", lw=0.7)
    ax.set_ylim(-0.05, 1.05); ax.set_ylabel("μ(rough)")
    ax.legend(fontsize=8); ax.grid(True, ls=":", alpha=0.5)

axes[-1].set_xlabel("step")
plt.suptitle("Router comparison — μ(rough model) over time", fontsize=12)
plt.tight_layout(); plt.show()


## 10 — Benchmark: when Gaussian likelihoods get it wrong

The **BayesianRouter** updates mode probabilities using a Gaussian innovation likelihood:

$$\mu_j \;\propto\; c_j \cdot L_j, \qquad L_j = \mathcal{N}(\nu_j;\,0,\,S_j)$$

This is optimal when all models are correctly specified and innovations are truly Gaussian.
In practice, real-world noise is often **heavy-tailed** — outlier measurements occur far more
frequently than the Gaussian assumption predicts.

### The heavy-tailed benchmark

| | Gaussian regime (A) | Heavy-tailed regime (B) |
|---|---|---|
| Process noise | $w_k \sim \mathcal{N}(0,\,Q_A)$ | $w_k \sim t_{\nu=3}$ scaled to $\text{Var}=Q_B$ |
| Tail behaviour | Drops as $e^{-x^2}$ | Drops as $x^{-4}$ — orders of magnitude heavier |
| Kalman model A | $Q = Q_A$ — calibrated | NIS spikes at outlier steps |
| Kalman model B | over-conservative | $Q = Q_B$ — correct variance, **wrong shape** |

**Why Bayesian struggles with heavy tails:**

The Gaussian PDF $L_j = \mathcal{N}(\nu;\,0,\,S_j)$ drops off exponentially for large $|\nu|$.
When a Student-$t$ outlier arrives, Bayesian assigns an extremely low likelihood to
**both** models, but the ratio $L_A / L_B$ becomes erratic because the Gaussian assumption
amplifies small differences in $S$ into huge likelihood swings at the tails.

Result: Bayesian **overreacts** to outlier innovations, producing noisy $\mu$ oscillations
even when the correct regime hasn't changed.

**Why NN routers improve:**

A neural network trained on a sliding window of NIS values learns the **empirical**
relationship between NIS patterns and the true regime — no Gaussian assumption required.
Heavy-tailed NIS values (high excess kurtosis) become a learnable feature rather than a
source of miscalibration.

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt
from scipy.stats import t as t_dist
from kalman.filters import InteractiveKalmanFilter
from kalman.framework.filter import KalmanFilter
from kalman.framework.routers import BayesianRouter, ThresholdRouter, NeuralRouter
from kalman.framework.routers.nn import MLP, CNN, LSTM, TransformerNN

np.random.seed(42)

# ── Parameters ────────────────────────────────────────────────────────────────
Q_A   = 0.1      # Gaussian regime: gentle process noise
Q_B   = 2.0      # heavy-tailed regime: higher variance
DF    = 3        # Student-t degrees of freedom (excess kurtosis = ∞ for df≤4)
R_OBS = 0.5      # observation noise (same for both models)
WINDOW = 15      # NIS sliding window for NN routers

# ── Models & IKF factory ──────────────────────────────────────────────────────
def _make_models():
    kf_a = KalmanFilter(n=1, m=1,
                        F=np.array([[1.]]), H=np.array([[1.]]),
                        Q=np.array([[Q_A]]), R=np.array([[R_OBS]]),
                        x0=np.zeros((1, 1)), P0=2 * np.eye(1))
    kf_b = KalmanFilter(n=1, m=1,
                        F=np.array([[1.]]), H=np.array([[1.]]),
                        Q=np.array([[Q_B]]), R=np.array([[R_OBS]]),
                        x0=np.zeros((1, 1)), P0=2 * np.eye(1))
    return [kf_a._model, kf_b._model]

Pi = np.array([[0.85, 0.15], [0.15, 0.85]])

def make_ikf(router=None):
    return InteractiveKalmanFilter(_make_models(), Pi, router=router)

# ── Data generation ───────────────────────────────────────────────────────────
def gen_segment(regime, T, rng=None):
    """Regime 0: Gaussian noise.  Regime 1: Student-t(3) noise (same variance)."""
    rng = rng or np.random.default_rng()
    x, zk = 0., []
    for k in range(T):
        if regime == 0:
            w = rng.normal(0, Q_A ** .5)
        else:
            scale = (Q_B * (DF - 2) / DF) ** .5
            w = float(t_dist.rvs(DF, scale=scale, random_state=rng))
        x += w
        zk.append([x + rng.normal(0, R_OBS ** .5)])
    return np.array(zk), np.full(T, regime, dtype=int)

def gen_switching(T, seg_len, seed=None):
    rng = np.random.default_rng(seed)
    zparts, lparts = [], []
    for s in range(T // seg_len):
        r = s % 2
        z, l = gen_segment(r, seg_len, rng=rng)
        zparts.append(z); lparts.append(l)
    return np.concatenate(zparts), np.concatenate(lparts)

# ── Part 1 — visualise NIS patterns ──────────────────────────────────────────
T_vis = 80
zk_A, _ = gen_segment(0, T_vis, rng=np.random.default_rng(7))
zk_B, _ = gen_segment(1, T_vis, rng=np.random.default_rng(7))

nis_per_regime = {}
for label, zk in [("Gaussian (A)", zk_A), ("Student-t (B)", zk_B)]:
    ikf_v = make_ikf(); ikf_v.forward(zk)
    nis_per_regime[label] = np.array(ikf_v.nis_log)

fig, axes = plt.subplots(2, 2, figsize=(15, 6), sharex=True)
sp = np.arange(T_vis)

for col, label in enumerate(["Gaussian (A)", "Student-t (B)"]):
    nis = nis_per_regime[label]
    for row, (mid, color, mname) in enumerate([(0, "steelblue", "Model A (Q=0.1)"),
                                                (1, "darkorange", "Model B (Q=2.0)")]):
        ax = axes[row, col]
        ax.plot(sp, nis[:, mid], color=color, lw=1.2, label=f"{mname} NIS")
        ax.axhline(1.0, color="black", ls="--", lw=0.7, label="$\\chi^2(1)$ mean")
        ax.set_title(f"{mname} NIS — {label}", fontsize=10)
        ax.set_ylabel("NIS")
        ax.set_ylim(-0.2, max(15, nis[:, mid].max() * 1.05))
        ax.legend(fontsize=8); ax.grid(True, ls=":", alpha=0.5)
        if row == 1:
            ax.set_xlabel("step")

fig.suptitle("NIS patterns: Gaussian vs Student-t regime\n"
             "Heavy tails produce occasional extreme NIS spikes that "
             "miscalibrate Gaussian likelihoods", fontsize=11)
plt.tight_layout(); plt.show()

# NIS statistics
print("NIS statistics per regime per model:")
for label in ["Gaussian (A)", "Student-t (B)"]:
    nis = nis_per_regime[label]
    for j, mname in enumerate(["Model A", "Model B"]):
        n = nis[:, j]
        kurt = float(np.mean((n - n.mean())**4) / n.std()**4 - 3)
        print(f"  {label}, {mname}: mean={n.mean():.2f}  std={n.std():.2f}  excess kurtosis={kurt:.0f}")
print(f"\nExcess kurtosis >> 0 in regime B confirms heavy tails in the NIS distribution.")
print(f"Bayesian's Gaussian likelihood L = N(ν; 0, S) is systematically wrong at these tails.")

In [ ]:
# ── Part 2 — train all routers ────────────────────────────────────────────────
print("Generating 16 training sequences (400 steps each) ...")
train_zk, train_lbl = [], []
for i in range(16):
    z, l = gen_switching(400, 80, seed=i)
    train_zk.append(z); train_lbl.append(l)
print(f"  Total training steps: {sum(len(l) for l in train_lbl)}")
print()

all_routers = {
    "Bayesian":   BayesianRouter(),
    "Threshold":  ThresholdRouter(m=1, window=10, soft=True),
}

neural_specs = [
    ("MLP",         MLP(hidden=48),                     dict(epochs=200, lr=2e-3, seed=0)),
    ("CNN",         CNN(out_channels=16, kernel_size=4), dict(epochs=200, lr=3e-3, seed=1)),
    ("LSTM",        LSTM(hidden=24),                    dict(epochs=300, lr=1e-3, seed=2)),
    ("Transformer", TransformerNN(d_model=16),          dict(epochs=200, lr=2e-3, seed=3)),
]

print("Training neural routers ...")
for name, arch, kw in neural_specs:
    t0 = time.time()
    router = NeuralRouter(model=arch, window=WINDOW)
    ikf_tr = make_ikf(router=router)
    ikf_tr.fit(train_zk, train_lbl, **kw)
    all_routers[name] = router
    print(f"  {name:<12} trained in {time.time() - t0:.0f}s")
print()

# ── Part 3 — evaluate on held-out sequences ──────────────────────────────────
def evaluate(router, n_seqs=20, seed_base=500):
    ok = tot = 0
    for i in range(n_seqs):
        z, l = gen_switching(300, 60, seed=seed_base + i)
        ikf_e = make_ikf(router=router)
        ikf_e.forward(z)
        preds = np.argmax(np.array(ikf_e.mu_log), axis=1)
        ok += int((preds == l).sum()); tot += len(l)
    return ok / tot

print("Evaluating on 20 held-out test sequences:")
results = {name: evaluate(r) for name, r in all_routers.items()}
for name, acc in results.items():
    print(f"  {name:<12}: {acc:.1%}")

best_nn = max(results[n] for n in ["MLP", "CNN", "LSTM", "Transformer"])
print(f"\nBest NN gain over Bayesian: +{(best_nn - results['Bayesian'])*100:.1f} pp")

# ── Part 4 — μ trajectory comparison ─────────────────────────────────────────
z_test, l_test = gen_switching(300, 60, seed=999)
show_routers = ["Bayesian", "Threshold", "CNN", "Transformer"]
colors_map = {"Bayesian": "steelblue", "Threshold": "seagreen",
              "CNN": "crimson", "Transformer": "purple"}

mu_traces = {}
for rname in show_routers:
    ikf_t = make_ikf(router=all_routers[rname])
    ikf_t.forward(z_test)
    mu_traces[rname] = np.array(ikf_t.mu_log)

fig, axes = plt.subplots(len(show_routers) + 1, 1,
                          figsize=(15, 3 * (len(show_routers) + 1)), sharex=True)
t_sp = np.arange(len(z_test))

axes[0].scatter(t_sp, z_test, s=5, alpha=0.25, color="tab:orange", label="measurements")
axes[0].fill_between(t_sp, z_test.min() - 1, z_test.max() + 1,
                     where=(l_test == 1), alpha=0.15, color="crimson",
                     label="Student-t regime (true)")
axes[0].set_ylabel("z [a.u.]")
axes[0].set_title("Observations + true regime (red = heavy-tailed)")
axes[0].legend(fontsize=8); axes[0].grid(True, ls=":", alpha=0.5)

for ax, rname in zip(axes[1:], show_routers):
    mu = mu_traces[rname]
    acc_test = float((np.argmax(mu, axis=1) == l_test).mean())
    ax.plot(t_sp, mu[:, 1], color=colors_map[rname], lw=1.5,
            label=f"{rname}  (acc = {acc_test:.1%})")
    ax.fill_between(t_sp, 0, l_test, alpha=0.10, color="crimson")
    ax.axhline(0.5, color="black", ls="--", lw=0.7, label="decision boundary")
    ax.set_ylim(-0.05, 1.05); ax.set_ylabel("$\\mu$(heavy-tailed)")
    ax.legend(fontsize=8); ax.grid(True, ls=":", alpha=0.5)

axes[-1].set_xlabel("step")
fig.suptitle("$\\mu$(heavy-tailed model) over time\n"
             "Bayesian oscillates on outlier innovations; "
             "CNN/Transformer give smoother, more accurate routing", fontsize=11)
plt.tight_layout(); plt.show()

# ── Part 5 — accuracy bar chart ──────────────────────────────────────────────
bar_colors = ["steelblue", "seagreen", "darkorange", "crimson", "darkgreen", "purple"]
names = list(results.keys())
accs  = [results[n] * 100 for n in names]

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(names, accs, color=bar_colors[:len(names)], alpha=0.85, zorder=3)
for bar, a in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{a:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.axhline(50, color="red", ls=":", lw=1.3, label="random baseline (50%)")
ax.set_ylabel("classification accuracy [%]", fontsize=11)
ax.set_title("Router accuracy: Gaussian vs Student-t regime detection\n"
             "Heavy tails miscalibrate Bayesian; NN routers learn the empirical pattern",
             fontsize=11)
ax.set_ylim(0, 108); ax.legend(fontsize=9)
ax.grid(True, axis="y", ls=":", alpha=0.6, zorder=0)
plt.tight_layout(); plt.show()

print("\n── Summary ──────────────────────────────────────────────────────────────")
print(f"{'Router':<14} {'Accuracy':>10}  {'vs Bayesian':>12}")
print("-" * 40)
bay = results["Bayesian"]
for name, acc in results.items():
    delta = f"+{(acc - bay)*100:.1f} pp" if acc > bay else f"{(acc - bay)*100:.1f} pp"
    print(f"{name:<14} {acc:>10.1%}  {delta:>12}")
print()
print("The Gaussian-likelihood assumption (Bayesian) becomes a liability when")
print("the true noise has heavy tails.  NN routers trained on actual NIS patterns")
print("learn to handle the excess kurtosis without this structural limitation.")

## Summary

### Control layer: P → PID → LQR → Neural

| Controller | Strategy | Training? |
|---|---|---|
| `ProportionalKalmanController` | u = −Kp · e | None |
| `PIDKalmanController` | u = −Kp e − Ki ∫e − Kd ė | None |
| `LQRKalmanController` | Optimal gain from DARE | `fit()` (analytical) |
| `NeuralKalmanController` | MLP policy via MPPI | `fit_mppi()` on `model.f()` |

### IMM routing layer

The `InteractiveKalmanFilter` is agnostic to how mode probabilities are computed.
Any router with the signature `mu = router(nis_arr, c, likelihoods)` can be plugged in.
`NeuralRouter` owns NIS feature extraction only — the `model` (e.g. `MLP`) is a separate,
pluggable architecture following the `model(features) → probs` convention:

| Router | Strategy | Training? |
|---|---|---|
| `BayesianRouter()` | μ_j ∝ c_j · N(ν_j; 0, S_j) — default | None |
| `ThresholdRouter()` | Assign model j closest to NIS = m | None |
| `NeuralRouter(model=MLP())` | model(NIS window) → probs | `ikf.fit(...)` |
| `HybridRouter(r_a, r_b, α)` | r_a^(1−α) · r_b^α geometric blend | trains both children |

### Adaptive estimation

| Module | Strategy | Training? |
|---|---|---|
| `BoostKalmanController` | Shrinks R when NIS >> m | None |
| `AdaptiveQRKalmanController` | Scales Q from empirical innovation covariance | None |

### Architecture

```
InteractiveKalmanFilter:          KalmanController:
  state mixing       ┐              u = ctrl(x̂, P, t, ν)
  predict + update   ├─ estimation   ↓
  NIS computation    ┘            u feeds back into predict step

  router.route(nis_arr, c, L) → μ   # routing is fully decoupled from estimation
  x̂ = Σ μ_j x̂_j                    # combined estimate
```

- **Estimation** (F, H, K, gain) is never modified — stays classical and optimal.
- **Control** outputs u only — sees x̂ − x_ref, diag(P), ν; never F, H, raw z.
- **Routing** is a pluggable strategy — swap without touching the filter.
- **NIS windows avoid distribution shift** — NIS depends only on filter predictions,
  not on which router chose μ, so training data generalises to any inference router.
